<a href="https://colab.research.google.com/github/soheldatta17/meta_model_knee/blob/main/Predict_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import pickle
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

import subprocess

try:
    import gdown
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "gdown", "-q"], check=True)
    import gdown

DRIVE_ASSETS = {
    "bundle": {
        "id": "1asiAmtlq5t3dcBfv56kUOWQfUlOgaEv8",
        "local": "/content/full_pipeline_bundle.pkl",
        "desc": "Pipeline bundle (XGBoost + scaler + head weights)",
    }
}


def download_bundle():
    asset = DRIVE_ASSETS["bundle"]
    local = asset["local"]

    if os.path.exists(local):
        print(f"[cached] {asset['desc']}")
        return local

    print(f"[download] {asset['desc']} ...")

    gdown.download(
        f"https://drive.google.com/uc?id={asset['id']}",
        local,
        quiet=False,
        fuzzy=True,
    )

    if not os.path.exists(local):
        raise RuntimeError("Bundle download failed")

    print(f"[saved] {local}")
    return local


# ---------------------------------------------------------------------------
# Config  (overridable via env vars or CLI flags)
# ---------------------------------------------------------------------------
API_URL     = os.getenv("API_URL",     "https://solure-knee-oa-feature-service.hf.space")
BUNDLE_PATH = os.getenv("BUNDLE_PATH", "/content/full_pipeline_bundle.pkl")
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

# Set to True to print raw API responses, internal vectors, and technical
# debug info. Leave False for a clean, easy-to-read result.
# (In Colab, just edit this line directly — no env var needed.)
VERBOSE = False


# ---------------------------------------------------------------------------
# KLGradeHead  (must match architecture used during training exactly)
# ---------------------------------------------------------------------------
class KLGradeHead(nn.Module):
    def __init__(self, in_dim: int = 512, hidden: int = 256, num_cls: int = 5):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_dim),
            nn.Dropout(0.4),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.BatchNorm1d(hidden),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, num_cls),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ---------------------------------------------------------------------------
# Bundle loading
# ---------------------------------------------------------------------------
_bundle_cache: dict | None = None


def load_bundle(path: str = BUNDLE_PATH) -> dict:
    """Load and cache the pipeline bundle (XGBoost + scaler + KLGradeHead weights)."""
    global _bundle_cache
    if _bundle_cache is not None:
        return _bundle_cache

    p = Path(path)
    if not p.exists():
        print(f"[ERROR] Bundle not found: {p}")
        print("  Place full_pipeline_bundle.pkl next to predict.py, or set BUNDLE_PATH.")
        sys.exit(1)

    # Force CPU map_location so head weights load on CPU-only machines
    _orig = torch.load
    def _cpu_load(f, *a, **kw):
        kw.setdefault("map_location", "cpu")
        kw.setdefault("weights_only", False)
        return _orig(f, *a, **kw)
    torch.load = _cpu_load
    try:
        with open(p, "rb") as f:
            bundle = pickle.load(f)
    finally:
        torch.load = _orig

    # Rebuild KLGradeHead from saved weights
    cfg  = bundle["head_config"]
    head = KLGradeHead(cfg["in_dim"], cfg["hidden"], cfg["num_cls"])
    head.load_state_dict(bundle["head_state_dict"])
    head.eval().to(DEVICE)
    bundle["_head_model"] = head

    print(f"Model ready (v{bundle['version']})")
    if VERBOSE:
        print(f"[bundle] grades  : {bundle['class_ids']}")
        print(f"[bundle] features: {bundle['feature_names']}")
        print(f"[bundle] head_config: {cfg}")
        print(f"[bundle] device     : {DEVICE}")

    _bundle_cache = bundle
    return bundle


# ---------------------------------------------------------------------------
# Feature extraction  —  calls the deployed API
# ---------------------------------------------------------------------------
def fetch_features(image_path: str, api_url: str = API_URL, timeout: int = 120) -> dict:
    """
    POST the image to /extract-features and return the full API response dict.

    Response shape:
      {
        "success": true,
        "features": {
          "vfusion": [...11 floats...],
          "backbone_embedding": [...512 floats...]
        },
        "model_outputs": {
          "severity_probs": [...5...],
          "jsn_probs": [...2...],
          "morph_probs": [...4...]
        },
        "metadata": { "inference_time_seconds": 0.47, ... }
      }
    """
    url = f"{api_url.rstrip('/')}/extract-features"
    p   = Path(image_path)
    ext = p.suffix.lower()
    mime = "image/png" if ext == ".png" else "image/jpeg"

    print(f"  -> POST {url}  [{p.name}]", end="", flush=True)
    t0 = time.perf_counter()

    with open(p, "rb") as f:
        resp = requests.post(url, files={"image": (p.name, f, mime)}, timeout=timeout)

    elapsed = time.perf_counter() - t0
    print(f"  -> {resp.status_code} ({elapsed:.2f}s)")

    if VERBOSE:
        print(f"  [response] status_code : {resp.status_code}")
        print(f"  [response] headers     : {dict(resp.headers)}")
        print(f"  [response] elapsed_s   : {elapsed:.4f}")
        print(f"  [response] raw_text    : {resp.text}")

    if resp.status_code != 200:
        raise RuntimeError(f"API HTTP {resp.status_code}: {resp.text[:300]}")

    data = resp.json()

    if VERBOSE:
        print("  [response] parsed JSON :")
        print(json.dumps(data, indent=2, default=str))

    if not data.get("success"):
        raise RuntimeError(f"API returned success=false: {data}")
    return data


# ---------------------------------------------------------------------------
# Downstream inference  —  KLGradeHead + XGBoost
# ---------------------------------------------------------------------------
def run_downstream(api_response: dict, bundle: dict) -> dict:
    """
    Takes the raw API response and runs the full downstream pipeline.

    Steps:
      1. backbone (512D) -> KLGradeHead -> head_probs (5D)
      2. concat(vfusion(11D), head_probs(5D)) = 16D
      3. StandardScaler -> XGBoost -> KL grade
    """
    vfusion  = np.array(api_response["features"]["vfusion"],           dtype=np.float32)
    backbone = np.array(api_response["features"]["backbone_embedding"], dtype=np.float32)
    sev_probs   = np.array(api_response["model_outputs"]["severity_probs"], dtype=np.float32)
    jsn_probs   = np.array(api_response["model_outputs"]["jsn_probs"],      dtype=np.float32)
    morph_probs = np.array(api_response["model_outputs"]["morph_probs"],    dtype=np.float32)

    if VERBOSE:
        print("\n  [downstream] input shapes:")
        print(f"    vfusion     : {vfusion.shape}  -> {vfusion.tolist()}")
        print(f"    backbone    : {backbone.shape}")
        print(f"    sev_probs   : {sev_probs.shape}  -> {sev_probs.tolist()}")
        print(f"    jsn_probs   : {jsn_probs.shape}  -> {jsn_probs.tolist()}")
        print(f"    morph_probs : {morph_probs.shape}  -> {morph_probs.tolist()}")

    # KLGradeHead: 512D -> 5D
    head: KLGradeHead = bundle["_head_model"]
    with torch.no_grad():
        bb_t       = torch.tensor(backbone[None], dtype=torch.float32).to(DEVICE)
        head_logits = head(bb_t)
        head_probs  = F.softmax(head_logits, dim=1).cpu().numpy()[0]

    if VERBOSE:
        print(f"\n  [downstream] KLGradeHead logits : {head_logits.cpu().numpy().tolist()}")
        print(f"  [downstream] KLGradeHead probs  : {head_probs.tolist()}")

    # XGBoost: vfusion(11) + head_probs(5) = 16D
    xgb_raw   = np.hstack([vfusion, head_probs])[None]
    xgb_input = bundle["scaler"].transform(xgb_raw)

    if VERBOSE:
        print(f"\n  [downstream] xgb_raw (pre-scale)  : {xgb_raw.tolist()}")
        print(f"  [downstream] xgb_input (scaled)   : {xgb_input.tolist()}")

    xgb       = bundle["xgb"]
    grade_idx = int(xgb.predict(xgb_input)[0])
    proba     = xgb.predict_proba(xgb_input)[0]

    if VERBOSE:
        print(f"  [downstream] xgb predicted index  : {grade_idx}")
        print(f"  [downstream] xgb class probabilities : {proba.tolist()}")

    class_ids      = bundle["class_ids"]
    severity_names = bundle["severity_names"]

    result = {
        "kl_grade"  : class_ids[grade_idx],
        "severity"  : severity_names[grade_idx],
        "confidence": round(float(proba[grade_idx]) * 100, 2),
        "grade_probabilities": {
            f"G{class_ids[i]} {severity_names[i]}": round(float(proba[i]) * 100, 2)
            for i in range(bundle["num_classes"])
        },
        "vectors": {
            "vfusion"      : vfusion.tolist(),
            "backbone"     : backbone.tolist(),
            "head_probs"   : head_probs.tolist(),
            "severity_probs": sev_probs.tolist(),
            "jsn_probs"    : jsn_probs.tolist(),
            "morph_probs"  : morph_probs.tolist(),
        },
        "api_inference_time_s": api_response["metadata"]["inference_time_seconds"],
    }

    if VERBOSE:
        print("\n  [downstream] final result dict:")
        print(json.dumps({k: v for k, v in result.items() if k != "vectors"}, indent=2, default=str))

    return result


# ---------------------------------------------------------------------------
# High-level predict
# ---------------------------------------------------------------------------
def predict(image_path: str, bundle: dict | None = None, api_url: str = API_URL) -> dict:
    """Full pipeline: image file -> KL grade dict."""
    if bundle is None:
        bundle = load_bundle()
    result       = run_downstream(fetch_features(image_path, api_url=api_url), bundle)
    result["file"] = Path(image_path).name
    return result


# ---------------------------------------------------------------------------
# Pretty-print
# ---------------------------------------------------------------------------
def print_result(r: dict) -> None:
    """Concise technical summary of the prediction."""
    print(f"  File          : {r['file']}")
    print(f"  Predicted KL  : {r['kl_grade']} ({r['severity']})")
    print(f"  Confidence    : {r['confidence']:.2f}%")
    print(f"  API inference : {r['api_inference_time_s']}s")
    print()
    print("  Class probabilities:")
    for label, pct in r["grade_probabilities"].items():
        bar = chr(9608) * int(pct / 2)
        print(f"    {label:<14} {pct:6.2f}%  {bar}")

    if VERBOSE:
        print()
        print("  Sub-model outputs (from API):")
        sev_names = ["Normal", "Doubtful", "Mild", "Moderate", "Severe"]
        sev_p   = r["vectors"]["severity_probs"]
        jsn     = r["vectors"]["jsn_probs"]
        morph   = r["vectors"]["morph_probs"]
        head    = r["vectors"]["head_probs"]
        print("    severity_probs :", {sev_names[i]: round(sev_p[i], 4) for i in range(5)})
        print("    jsn_probs      :", {"P(narrow)": round(jsn[0], 4), "P(normal)": round(jsn[1], 4)})
        print("    morph_probs    :", [round(v, 4) for v in morph])
        print("    head_probs     :", [round(v, 4) for v in head])
        print()
        print("  Full vectors (raw):")
        print(json.dumps(r["vectors"], indent=2, default=str))


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def _is_notebook() -> bool:
    """True when running inside Jupyter / Google Colab."""
    try:
        shell = get_ipython().__class__.__name__  # type: ignore[name-defined]
        return shell in ("ZMQInteractiveShell", "Shell")   # Jupyter / Colab
    except NameError:
        return False


def main() -> None:
    # ── Colab / Jupyter: argparse would choke on kernel args like -f ─────────
    if _is_notebook():
        # Edit these two lines directly when running in a notebook
        notebook_input  = f"/content/{random.randint(0, 4)}.png"
        notebook_api    = API_URL
        notebook_bundle = BUNDLE_PATH

        bundle     = load_bundle(notebook_bundle)
        input_path = Path(notebook_input)

        images = (
            sorted(p for p in input_path.iterdir() if p.suffix.lower() in SUPPORTED_EXTS)
            if input_path.is_dir()
            else [input_path]
        )

        print(f"\nAPI    : {notebook_api}")
        print(f"Bundle : {notebook_bundle}")
        print(f"Images : {len(images)}")
        print("=" * 60)

        passed = failed = 0
        for img_path in images:
            print(f"\n[{img_path.name}]")
            try:
                print_result(predict(str(img_path), bundle, api_url=notebook_api))
                passed += 1
            except Exception as exc:
                print(f"  ERROR: {exc}")
                failed += 1
            print("-" * 60)

        print(f"\nDone — {passed} succeeded, {failed} failed.")
        return

    # ── Normal CLI (terminal / script) ───────────────────────────────────────
    parser = argparse.ArgumentParser(
        description="Knee OA KL-Grade predictor.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument("input", help="Image file or folder of images.")
    parser.add_argument("--api-url", default=API_URL,
                        help=f"Feature extraction API URL (default: {API_URL})")
    parser.add_argument("--bundle", default=BUNDLE_PATH,
                        help=f"Path to full_pipeline_bundle.pkl (default: {BUNDLE_PATH})")
    args = parser.parse_args()

    bundle     = load_bundle(args.bundle)
    input_path = Path(args.input)

    if input_path.is_dir():
        images = sorted(p for p in input_path.iterdir()
                        if p.suffix.lower() in SUPPORTED_EXTS)
        if not images:
            print(f"No supported images found in {input_path}")
            sys.exit(1)
    elif input_path.is_file():
        images = [input_path]
    else:
        print(f"[ERROR] Path not found: {input_path}")
        sys.exit(1)

    print(f"\nAPI    : {args.api_url}")
    print(f"Bundle : {args.bundle}")
    print(f"Images : {len(images)}")
    print("=" * 60)

    passed = failed = 0
    for img_path in images:
        print(f"\n[{img_path.name}]")
        try:
            print_result(predict(str(img_path), bundle, api_url=args.api_url))
            passed += 1
        except Exception as exc:
            print(f"  ERROR: {exc}")
            failed += 1
        print("-" * 60)

    print(f"\nDone — {passed} succeeded, {failed} failed.")


if __name__ == "__main__":
    download_bundle()
    main()

[cached] Pipeline bundle (XGBoost + scaler + head weights)
Model ready (vv4.0)

API    : https://solure-knee-oa-feature-service.hf.space
Bundle : /content/full_pipeline_bundle.pkl
Images : 1

[1.png]
  -> POST https://solure-knee-oa-feature-service.hf.space/extract-features  [1.png]